# 🏛️ MEF · Canon y sobrecanon petrolero recaudado por cada municipalidad (2005–hoy)

PERUPETRO publica el canon **por municipalidad** solo en 1993–2002 (Excel de Distribución) y el año en curso (Excel de Canon Municipalidades). Desde 2003 entrega el total municipal a un intermediario (CND hasta 2006; PCM desde 2007), que hace el reparto. Este notebook completa los años que faltan con el **MEF – Consulta Amigable de Ingresos** (`apps5.mineco.gob.pe/transparenciaingresos/mensual/`), monto **recaudado** por cada municipalidad:

**Ruta:** Fuente 5 (Recursos determinados) → Rubro 18 → Tipo de recurso con «PETROLERO» (5: Canon y sobrecanon petrolero y, en años recientes, K: Sub cuenta – Canon y sobrecanon petrolero; se suman) → Nivel de gobierno M → Departamento 16: Loreto → Municipalidad.

- La consulta funciona con *postbacks* de ASP.NET (no se puede armar por URL), así que se maneja un **navegador real** (Playwright) con pausas, como lo haría una persona.
- **Desde 2007** la consulta es homogénea (fuente 5 → rubro 18 → tipos petroleros 5 y K).
- **2005–2006**: otra clasificación (fuente 01 Canon y sobrecanon → tipo de recurso, sin rubro) y el detalle por municipalidad solo está en el tipo «0: NORMAL», que reúne **todos** los canones (no solo el petrolero). Con `SI_NO_HAY_PETROLERO = "Usar NORMAL"` (opción por defecto) se toma ese tipo cuando el petrolero no llega a gobiernos locales; queda rotulado «todos los canones» en el CSV y en el mapa.
- **2003–2004**: sin detalle por municipalidad (el canon petrolero municipal figura a nombre del Consejo Nacional de Descentralización); quedan en el control con el motivo.
- **Resultado:** `DATA/PP_canon_municipal_mef_anio.csv` y `CONTROL/CONTROL PP canon_municipal_mef.csv` (suma de municipalidades vs. total de Loreto en el MEF, y vs. el total municipal de PERUPETRO del mismo año: *recaudado* y *transferido* pueden diferir por fechas de registro).
- El mapa (`PP_Mapa_Lotes_Plus.ipynb`) lo usa para los años sin detalle de PERUPETRO y lo rotula con su fuente.

**Bloque 1** instala el navegador (1–2 min). **Bloque 2** descarga y controla.

In [ ]:
#@title Bloque 1 · Instalar el navegador (Playwright + Chromium) { display-mode: "form" }
!pip -q install playwright
!playwright install --with-deps chromium > /dev/null 2>&1 && echo "✅ Chromium listo"


In [ ]:
#@title Bloque 2 · Descargar y controlar { display-mode: "form" }
ANIO_INICIO = 2003  #@param {type:"integer"}
ANIO_FIN = 2026  #@param {type:"integer"}
SI_NO_HAY_PETROLERO = "Usar NORMAL (todos los canones)"  #@param ["Usar NORMAL (todos los canones)", "Dejar sin dato"]
DEPARTAMENTO = "16: LORETO"  #@param ["TODOS", "16: LORETO", "25: UCAYALI", "20: PIURA", "24: TUMBES", "10: HUANUCO", "08: CUSCO"]
import os, re, time, asyncio, unicodedata, pandas as pd
from datetime import date
DRIVE_FOLDER_ID = "1dOOcsVyw915yKg6BzSdICVfCpYTiyeaq"            # DATA
DRIVE_FOLDER_CONTROL_ID = "1TKjS2Wjpnf26oladjhqstmN5c9Irq-Aj"    # CONTROL
BASE = "https://apps5.mineco.gob.pe/transparenciaingresos/mensual"
FUENTE = "MEF, Consulta Amigable de Ingresos (recaudado)"
ruta = "/content/pp_mef"; os.makedirs(ruta, exist_ok=True)

# ── Google Drive ──
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
auth.authenticate_user()
drive = build("drive", "v3", cache_discovery=False)
def archivos_en(folder_id):
    out, token = {}, None
    while True:
        r = drive.files().list(q=f"'{folder_id}' in parents and trashed = false", fields="nextPageToken, files(id,name)",
                               pageSize=1000, pageToken=token, supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
        out.update({f["name"]: f["id"] for f in r.get("files", [])})
        token = r.get("nextPageToken")
        if not token:
            return out
def subir(ruta_local, folder_id, existentes, mime="text/csv"):
    nombre = os.path.basename(ruta_local)
    media = MediaFileUpload(ruta_local, mimetype=mime, resumable=True)
    if nombre in existentes:
        drive.files().update(fileId=existentes[nombre], media_body=media, supportsAllDrives=True).execute()
    else:
        existentes[nombre] = drive.files().create(body={"name": nombre, "parents": [folder_id]}, media_body=media,
                                                  fields="id", supportsAllDrives=True).execute()["id"]
def bajar(nombre, existentes):
    if nombre not in existentes:
        return None
    p = os.path.join(ruta, nombre)
    with open(p, "wb") as fh:
        dl = MediaIoBaseDownload(fh, drive.files().get_media(fileId=existentes[nombre], supportsAllDrives=True)); hecho = False
        while not hecho:
            _, hecho = dl.next_chunk()
    return pd.read_csv(p, encoding="utf-8-sig")
datos_drive, control_drive = archivos_en(DRIVE_FOLDER_ID), archivos_en(DRIVE_FOLDER_CONTROL_ID)

# ── Lector: canon y sobrecanon petrolero recaudado por cada municipalidad (MEF, Consulta Amigable de Ingresos) ──
# La consulta es ASP.NET con postbacks (Navegar_3.aspx?y=AAAA + __doPostBack): no se arma por URL como la de gasto,
# así que se maneja un navegador real (Playwright): elegir la fila (radio) y pulsar el botón de la dimensión siguiente.
import re, asyncio

DIMS = ("Fuente de Financiamiento", "Rubro", "Tipo de Recurso", "Nivel de Gobierno", "Gob.Loc./Mancom.", "Sector", "Pliego",
        "Unidad Ejecutora", "Departamento", "Provincia", "Municipalidad", "Tipo de Ingreso")

JS_ESTADO = r"""() => {
  const limpio = s => (s || '').replace(/\s+/g, ' ').trim();
  const num = s => { const t = limpio(s).replace(/,/g, ''); return /^-?\d+(\.\d+)?$/.test(t) ? parseFloat(t) : null; };
  const out = {dim: null, filas: [], migas: [], botones: []};
  const radios = [...document.querySelectorAll('input[type=radio]')];
  for (const tr of document.querySelectorAll('tr')) {
    const celdas = [...tr.children].filter(x => x.tagName === 'TD' || x.tagName === 'TH');
    const tx = celdas.map(c => limpio(c.innerText));
    if (!out.dim && tx.some(t => /^Recaudado/.test(t)) && !tr.querySelector('input[type=radio]'))
      out.dim = tx.find(t => t && !/^(PIA|PIM|Recaudado)/.test(t)) || null;
    const r = tr.querySelector('input[type=radio]');
    const etiqueta = tx.find(t => /^[0-9A-ZÑ]{1,3}(-\d+)?:\s/.test(t));
    const nums = tx.map(num).filter(v => v !== null);
    if (r && etiqueta) {
      let rec = nums.length ? nums[nums.length - 1] : null;
      const v = (r.value || '').split('/');              // «código/PIA/PIM/recaudado» con decimales exactos
      if (v.length >= 3 && !isNaN(parseFloat(v[v.length - 1]))) rec = parseFloat(v[v.length - 1]);
      out.filas.push({i: radios.indexOf(r), etiqueta, recaudado: rec});
    }
    else if (!r && etiqueta === undefined && tx.length) {
      const m = tx.find(t => /^(TOTAL|Fuente de Financiamiento|Rubro|Tipo de Recurso|Nivel de Gobierno|Gob\.Loc|Sector|Pliego|Departamento|Provincia)\b/.test(t));
      if (m && nums.length) out.migas.push({etiqueta: m, recaudado: nums[nums.length - 1]});
    }
  }
  out.botones = [...document.querySelectorAll('input[type=submit],input[type=button],button')].map(b => limpio(b.value || b.innerText)).filter(x => x);
  return out;
}"""

class MEFIngresos:
    def __init__(self, page, base, log=print, pausa=1.0):
        self.page, self.base, self.log, self.pausa = page, base.rstrip("/"), log, pausa
        self.frame = page.main_frame

    async def _esperar_recarga(self, accion, timeout=60000):
        await self.frame.evaluate("() => { window.__pp = 1; }")
        await accion()
        try:
            await self.frame.wait_for_function("() => window.__pp === undefined", timeout=timeout)
        except Exception:
            pass                                          # sin recarga (selección solo en el navegador)
        await self.frame.wait_for_load_state("domcontentloaded")
        await asyncio.sleep(self.pausa)

    async def estado(self):
        return await self.frame.evaluate(JS_ESTADO)

    async def diagnostico(self):
        """Qué ve el navegador: título, marcos y primeras líneas de cada uno (para entender un bloqueo o un cambio de página)."""
        partes = [f"título «{await self.page.title()}»"]
        for f in self.page.frames:
            try:
                txt = (await f.evaluate("() => (document.body ? document.body.innerText : '')"))[:160]
            except Exception as e:
                txt = f"(no se pudo leer: {type(e).__name__})"
            partes.append(f"marco {f.url[:90]} → {' '.join(txt.split())!r}")
        return " | ".join(partes)

    async def _buscar_tabla(self, espera=45):
        """Espera hasta `espera` segundos a que algún marco muestre filas con radio (la tabla vive en un iframe)."""
        for _ in range(int(espera / 1.5)):
            marcos = [f for f in self.page.frames if "navegar" in f.url.lower()] + [self.page.main_frame]
            for fr in marcos:
                try:
                    if await fr.locator("input[type=radio]").count():
                        self.frame = fr
                        return await self.estado()
                except Exception:
                    pass
            await asyncio.sleep(1.5)
        return None

    async def abrir_anio(self, anio):
        # primero la página principal (fija el año en la sesión y carga el iframe Navegar_…); si no, el marco directo
        for url in (f"{self.base}/default.aspx?y={anio}", f"{self.base}/Navegar_3.aspx?y={anio}"):
            await self.page.goto(url, wait_until="domcontentloaded", timeout=90000)
            try:
                await self.page.wait_for_load_state("networkidle", timeout=30000)
            except Exception:
                pass
            e = await self._buscar_tabla()
            if e and e["filas"]:
                return e
        raise RuntimeError(f"{anio}: la consulta no mostró filas. Lo que ve el navegador: {await self.diagnostico()}")

    async def elegir(self, patron):
        e = await self.estado()
        f = next((x for x in e["filas"] if re.search(patron, x["etiqueta"], re.I)), None)
        if f is None:
            return None
        radio = self.frame.locator("input[type=radio]").nth(f["i"])
        await self._esperar_recarga(lambda: radio.click(), timeout=1500)
        return f

    async def ir_a(self, dimension):
        e = await self.estado()
        if e["dim"] and e["dim"].lower().startswith(dimension.lower()):
            return True
        if dimension not in e["botones"]:
            return False
        b = self.frame.locator(f"input[value='{dimension}'], button:has-text('{dimension}')").first
        await self._esperar_recarga(lambda: b.click())
        e = await self.estado()
        return bool(e["dim"] and e["dim"].lower().startswith(dimension.lower()))

    # dos clasificaciones: desde 2007, fuente 5 (Recursos determinados) → rubro 18 → tipo de recurso;
    # hasta 2006, fuente 01 (Canon y sobrecanon) → tipo de recurso (4: petrolero, 0: NORMAL…)
    ESQUEMAS = ([(r"^5:\s*RECURSOS DETERMINADOS", "Rubro"), (r"^18:", "Tipo de Recurso")],
                [(r"^0?1:\s*CANON Y SOBRECANON", "Tipo de Recurso")])

    async def _hasta_departamentos(self, anio, tipo_patron, esquema=0):
        """Navega hasta la lista «Departamento» (gobiernos locales) de un tipo de recurso. Devuelve la ruta o un error."""
        await self.abrir_anio(anio)
        ruta = []
        for patron, siguiente in self.ESQUEMAS[esquema] + [(tipo_patron, "Nivel de Gobierno"), (r"^M:", None)]:
            f = await self.elegir(patron)
            if f is None:
                e = await self.estado()
                return ruta + [f"NO ENCONTRADO {patron} en «{e['dim']}»"], False
            ruta.append(f["etiqueta"])
            if siguiente is None:                          # nivel M: en años recientes aparece «Gob.Loc./Mancom.» antes del departamento
                if await self.ir_a("Gob.Loc./Mancom."):
                    g = await self.elegir(r"^M:")
                    if g: ruta.append(g["etiqueta"])
                if not await self.ir_a("Departamento"):
                    return ruta + ["SIN BOTÓN Departamento"], False
            elif not await self.ir_a(siguiente):
                return ruta + [f"SIN BOTÓN {siguiente}"], False
        return ruta, True

    async def departamentos(self, anio, tipo_patron, esquema=0):
        """Departamentos con recaudado de ese tipo de recurso en gobiernos locales: [(etiqueta, recaudado)]."""
        ruta, ok = await self._hasta_departamentos(anio, tipo_patron, esquema)
        if not ok:
            return [], ruta
        e = await self.estado()
        return [(f["etiqueta"], f["recaudado"]) for f in e["filas"] if (f["recaudado"] or 0) > 0], ruta

    async def municipalidades(self, anio, tipo_patron, dep_patron=r"^16:\s*LORETO", esquema=0):
        """Filas «Municipalidad» del departamento para un tipo de recurso. Devuelve (filas, total_departamento, ruta)."""
        ruta, ok = await self._hasta_departamentos(anio, tipo_patron, esquema)
        if not ok:
            return None, None, ruta
        f = await self.elegir(dep_patron)
        if f is None:
            return None, None, ruta + [f"NO ENCONTRADO {dep_patron} en «Departamento»"]
        ruta.append(f["etiqueta"])
        if not await self.ir_a("Municipalidad"):
            return None, None, ruta + ["SIN BOTÓN Municipalidad"]
        e = await self.estado()
        total = next((m["recaudado"] for m in e["migas"] if m["etiqueta"].startswith("Departamento")), None)
        return e["filas"], total, ruta

    async def tipos_petroleros(self, anio, patron=r"PETROLER"):
        """Tipos de recurso que cumplen el patrón («PETROLERO»: 5 y K desde 2007; 4 hasta 2006): [(etiqueta, recaudado, esquema)]."""
        for esquema, pasos in enumerate(self.ESQUEMAS):
            await self.abrir_anio(anio)
            ok = True
            for pat, siguiente in pasos:
                if await self.elegir(pat) is None or not await self.ir_a(siguiente):
                    ok = False; break
            if not ok:
                continue
            e = await self.estado()
            return [(f["etiqueta"], f["recaudado"], esquema) for f in e["filas"] if re.search(patron, f["etiqueta"], re.I)]
        return []

def nombre_municipalidad(etiqueta):
    """«01-301422: MUNICIPALIDAD PROVINCIAL DE MAYNAS - IQUITOS» → (código, nivel, municipalidad, provincia)."""
    cod, _, txt = etiqueta.partition(":")
    txt = txt.strip().upper()
    m = re.match(r"MUNICIPALIDAD\s+(PROVINCIAL|DISTRITAL)\s+(?:(?:DE\s+LA|DEL|DE)\s+)?(.*)$", txt)
    if not m:
        return cod.strip(), "", txt, ""
    nivel, nom = m.group(1), m.group(2).strip()
    if nivel == "PROVINCIAL":
        prov, _, capital = nom.partition(" - ")
        return cod.strip(), nivel, (capital or prov).strip(), prov.strip()
    return cod.strip(), nivel, nom, ""


def norm(t):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFKD", str(t or "")).encode("ascii", "ignore").decode()).strip().upper()
TODOS = DEPARTAMENTO.strip().upper() == "TODOS"      # todos los departamentos que el MEF muestre con canon petrolero municipal
# referencias de PERUPETRO (si existen en DATA): total municipal por año y provincia de cada municipalidad distrital
dist_pp = bajar("PP_canon_distribucion_mes.csv", datos_drive)
mun_pp = bajar("PP_canon_municipal_mes.csv", datos_drive)
pp_mun = (dist_pp[dist_pp["TIPO_BENEFICIARIO"] == "MUNICIPALIDADES"].assign(_D=dist_pp["DEPARTAMENTO"].map(norm))
          .groupby(["_D", "AÑO"])["MONTO_SOLES"].sum().to_dict() if dist_pp is not None else {})
prov_de = ({norm(m): norm(p) for m, p in zip(mun_pp["MUNICIPALIDAD"], mun_pp["PROVINCIA"]) if isinstance(p, str)}
           if mun_pp is not None else {})

from playwright.async_api import async_playwright
filas, ctrl = [], []
async def correr():
    async with async_playwright() as p:
        # navegador lo más parecido a uno normal (el portal tiene un filtro anti-bots)
        nav = await p.chromium.launch(args=["--disable-blink-features=AutomationControlled"])
        ctx_nav = await nav.new_context(locale="es-PE", viewport={"width": 1366, "height": 900},
                                        user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) "
                                                   "Chrome/124.0 Safari/537.36", extra_http_headers={"Accept-Language": "es-PE,es;q=0.9"})
        await ctx_nav.add_init_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
        pg = await ctx_nav.new_page()
        m = MEFIngresos(pg, BASE, pausa=1.5)
        fallas_seguidas = 0
        for anio in range(ANIO_INICIO, ANIO_FIN + 1):
            if fallas_seguidas >= 3:
                print(f"⛔ {anio}: se detiene tras 3 años seguidos con error; revisa el diagnóstico de arriba y la captura en CONTROL")
                ctrl.append({"AÑO": anio, "DEPARTAMENTO": "TODOS" if TODOS else DEPARTAMENTO, "ESTADO": "NO INTENTADO (3 años seguidos con error)"})
                continue
            for intento in (1, 2):
                try:
                    tipos = await m.tipos_petroleros(anio)
                    usa_normal = False
                    if SI_NO_HAY_PETROLERO.startswith("Usar") and anio < 2007:
                        # hasta 2006 el tipo petrolero puede no llegar a gobiernos locales (2003–2004: va al CND);
                        # en 2005–2006 el detalle municipal está en «0: NORMAL», que reúne todos los canones
                        con_m = False
                        for etq, _, esq in tipos:
                            ds, _r = await m.departamentos(anio, "^" + re.escape(etq.split(":")[0].strip()) + r":\s", esq)
                            if ds:
                                con_m = True; break
                        if not con_m:
                            tn = await m.tipos_petroleros(anio, r"^0:\s*NORMAL")
                            if tn:
                                tipos, usa_normal = tn, True
                    if not tipos:
                        ctrl.append({"AÑO": anio, "DEPARTAMENTO": "TODOS" if TODOS else DEPARTAMENTO, "ESTADO": "SIN TIPO DE RECURSO PETROLERO EN EL RUBRO 18 (¿detalle a nombre del CND?)"})
                        print(f"ℹ️ {anio}: sin tipo de recurso petrolero en el rubro 18"); break
                    if TODOS:                                  # departamentos con recaudado en algún tipo petrolero
                        deps = {}
                        for etq, _, esq in tipos:
                            ds, _r = await m.departamentos(anio, "^" + re.escape(etq.split(":")[0].strip()) + r":\s", esq)
                            for d_, v_ in ds:
                                deps[d_.split(":")[0].strip()] = d_
                        objetivos = sorted(deps.values())
                    else:
                        objetivos = [DEPARTAMENTO]
                    for dep_etq in objetivos:
                        cod_d, DEP = dep_etq.split(":", 1)[0].strip(), norm(dep_etq.split(":", 1)[1])
                        suma = tot = 0.0; n = 0; detalle = []
                        for etq, rec_nac, esq in tipos:
                            cod_t = etq.split(":")[0].strip()
                            fs, total, recorrido = await m.municipalidades(anio, "^" + re.escape(cod_t) + r":\s", r"^" + re.escape(cod_d) + r":", esq)
                            if fs is None:
                                detalle.append(f"{etq}: {' > '.join(recorrido)}"); continue
                            for f in fs:
                                if not re.search("MUNICIPALIDAD", f["etiqueta"], re.I):
                                    continue
                                cod, nivel, mu, prov = nombre_municipalidad(f["etiqueta"])
                                filas.append({"AÑO": anio, "DEPARTAMENTO": DEP, "CODIGO": cod, "MUNICIPALIDAD_MEF": f["etiqueta"].split(":", 1)[1].strip(),
                                              "MUNICIPALIDAD": mu, "NIVEL": nivel, "PROVINCIA": prov or prov_de.get(norm(mu), ""),
                                              "TIPO_RECURSO": etq + (" (todos los canones)" if usa_normal else ""), "RECAUDADO_SOLES": f["recaudado"] or 0.0,
                                              "FUENTE": FUENTE + (" · tipo NORMAL: todos los canones, no solo el petrolero" if usa_normal else "")})
                                suma += f["recaudado"] or 0.0; n += 1
                            tot += total or 0.0
                            detalle.append(f"{etq}: S/ {total or 0:,.0f}")
                        ref = pp_mun.get((DEP, anio))
                        ctrl.append({"AÑO": anio, "DEPARTAMENTO": DEP, "TIPOS": " | ".join(detalle), "MUNICIPALIDADES": n,
                                     "SUMA_MUNICIPALIDADES": round(suma, 2), "TOTAL_DEPARTAMENTO_MEF": round(tot, 2), "DIF": round(suma - tot, 2),
                                     "PERUPETRO_MUNICIPAL": round(ref, 2) if ref else None,
                                     "MEF_SOBRE_PERUPETRO": round(suma / ref, 4) if ref else None,
                                     "ESTADO": ("OK" if n and abs(suma - tot) <= 1 else "REVISAR" if n else "SIN MUNICIPALIDADES")})
                        print(f"{'✅' if n else 'ℹ️'} {anio} · {DEP}: {n} municipalidades · S/ {suma/1e6:,.2f} millones" + (f" · PERUPETRO S/ {ref/1e6:,.2f} millones" if ref else ""))
                    if not objetivos:
                        ctrl.append({"AÑO": anio, "DEPARTAMENTO": "TODOS" if TODOS else DEPARTAMENTO, "ESTADO": "SIN DEPARTAMENTOS CON CANON PETROLERO MUNICIPAL"})
                        print(f"ℹ️ {anio}: sin departamentos con canon petrolero en gobiernos locales")
                    fallas_seguidas = 0
                    break
                except Exception as e:
                    print(f"⚠️ {anio} intento {intento}: {type(e).__name__}: {str(e)[:700]}")
                    try:
                        png = os.path.join(ruta, f"CONTROL PP mef_error_{anio}.png"); await pg.screenshot(path=png, full_page=True)
                        subir(png, DRIVE_FOLDER_CONTROL_ID, control_drive, mime="image/png")
                    except Exception:
                        pass
                    if intento == 2:
                        fallas_seguidas += 1
                        ctrl.append({"AÑO": anio, "DEPARTAMENTO": "TODOS" if TODOS else DEPARTAMENTO, "ESTADO": f"ERROR: {type(e).__name__}: {str(e)[:150]}"})
                    await asyncio.sleep(10)
        await nav.close()
await correr()

# ── Guardar (se acumula: cada corrida reemplaza los años que vuelve a leer) ──
nuevo = pd.DataFrame(filas)
previo = bajar("PP_canon_municipal_mef_anio.csv", datos_drive)
if previo is not None and len(nuevo):
    leidos = set(zip(nuevo["DEPARTAMENTO"], nuevo["AÑO"]))
    previo = previo[[(d, a) not in leidos for d, a in zip(previo["DEPARTAMENTO"], previo["AÑO"])]]
    nuevo = pd.concat([previo, nuevo], ignore_index=True)
elif previo is not None:
    nuevo = previo
if len(nuevo):
    nuevo = nuevo.sort_values(["DEPARTAMENTO", "AÑO", "NIVEL", "MUNICIPALIDAD", "TIPO_RECURSO"], ascending=[True, True, False, True, True])
    p = os.path.join(ruta, "PP_canon_municipal_mef_anio.csv"); nuevo.to_csv(p, index=False, encoding="utf-8-sig")
    subir(p, DRIVE_FOLDER_ID, datos_drive)
c = pd.DataFrame(ctrl); c.insert(0, "FECHA_EXTRACCION", date.today().strftime("%d/%m/%Y"))
previo_c = bajar("CONTROL PP canon_municipal_mef.csv", control_drive)
if previo_c is not None:
    leidos_c = set(zip(c["DEPARTAMENTO"], c["AÑO"]))
    c = pd.concat([previo_c[[(d, a) not in leidos_c for d, a in zip(previo_c["DEPARTAMENTO"], previo_c["AÑO"])]], c],
                  ignore_index=True).sort_values(["AÑO", "DEPARTAMENTO"])
p = os.path.join(ruta, "CONTROL PP canon_municipal_mef.csv"); c.to_csv(p, index=False, encoding="utf-8-sig")
subir(p, DRIVE_FOLDER_CONTROL_ID, control_drive)
print(f"\n☁️ DATA/PP_canon_municipal_mef_anio.csv ({len(nuevo)} filas) · CONTROL/CONTROL PP canon_municipal_mef.csv")
display(c[c["AÑO"].between(ANIO_INICIO, ANIO_FIN)].drop(columns=["FECHA_EXTRACCION"]))
